# FiberNet v2.0: Graph-Based Fiber Network Tutorial

This tutorial demonstrates the complete **graph-based workflow** for fiber network research:

1. **Generate** structured fiber networks (Regular, ZigZag)
2. **Weld** crossing fibers to create realistic junctions
3. **Visualize** the network structure (fibers only, no node clutter)
4. **Extract** 94-dimensional feature vectors
5. **Build** ML surrogate models
6. **Explore** reinforcement learning for structure optimization

> **Note**: This tutorial uses the  graph representation introduced in v2.0.
> All graph operations work with  objects where nodes have a `pos` attribute
> and edges represent fiber segments.

## 1. Setup

Install FiberNet and import the required modules:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

import fibernet as fn
from fibernet.graph import weld_graph, find_intersections, save_graph_json, load_graph_json
from fibernet.gen import RegularNetworkGenerator, ZigZagGenerator
from fibernet.analysis import GraphFeatureExtractor
from fibernet.viz import plot_graph, plot_graph_comparison, plot_structure_stats

print(f"FiberNet v{fn.__version__}")
print(f"Available graph API: weld_graph, RegularNetworkGenerator, ZigZagGenerator, extract_features")

## 2. Generate a Regular Fiber Network

The `RegularNetworkGenerator` creates square-lattice based fiber networks,
inspired by real nonwoven manufacturing processes.

**Parameters:**
| Parameter | Default | Description |
|-----------|---------|-------------|
| `side_length` | 10 | Side length of the base square unit |
| `num_points_per_side` | 2 | Number of midpoints per side (more = denser fibers) |
| `perturbations` | None | List of (dx, dy) tuples for random displacement |
| `tiling` | 3 | Number of tiles in each direction (NxN array) |

In [ ]:
# Create a basic regular network generator
gen = RegularNetworkGenerator(
    side_length=10,
    num_points_per_side=2,
    tiling=3,
)

# Generate the graph
G = gen.generate()

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Quick visualization
fig, ax = plot_graph(G, title="Regular Network (3x3 tiles)")
plt.show()

## 3. Perturbation: Creating Realistic Disorder

Real fiber networks are never perfectly regular. The `perturbations` parameter
adds random displacement to fiber endpoints, simulating manufacturing variability.

Each perturbation tuple `(dx, dy)` defines the maximum displacement range
for random jitter on each side of the base unit.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
perturbations = [(0.0, 0.0), (0.5, 0.5), (1.0, 1.0), (2.0, 2.0)]

for i, perturb in enumerate(perturbations):
    gen = RegularNetworkGenerator(
        side_length=10,
        num_points_per_side=2,
        tiling=3,
        perturbations=[perturb],
    )
    G = gen.generate()
    
    # Manual plot for subplot
    pos = nx.get_node_attributes(G, 'pos')
    for u, v in G.edges():
        x = [pos[u][0], pos[v][0]]
        y = [pos[u][1], pos[v][1]]
        axes[i].plot(x, y, 'k-', linewidth=1.5)
    axes[i].set_title(f"perturb=({perturb[0]}, {perturb[1]})")
    axes[i].set_aspect('equal')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('perturbation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Perturbation increases structural disorder → affects mechanical properties")

## 4. Tiling: Building Large-Scale Networks

The `tiling` parameter controls the NxN array replication of the base unit.
Larger tilings create more realistic bulk material representations.

In [ ]:
tiling_sizes = [1, 2, 3, 5]
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

for i, t in enumerate(tiling_sizes):
    gen = RegularNetworkGenerator(side_length=10, num_points_per_side=1, tiling=t)
    G = gen.generate()
    
    pos = nx.get_node_attributes(G, 'pos')
    for u, v in G.edges():
        x = [pos[u][0], pos[v][0]]
        y = [pos[u][1], pos[v][1]]
        axes[i].plot(x, y, 'k-', linewidth=1.0)
    axes[i].set_title(f"tiling={t}x{t} ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")
    axes[i].set_aspect('equal')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('tiling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Weld Graph: Detecting and Welding Fiber Crossings

In real nonwoven materials, fibers **bond** where they cross. The `weld_graph()`
function detects all edge intersections and inserts junction nodes at crossing points.

**Algorithm:**
1. Test every pair of edges for geometric intersection (using Shapely)
2. Insert new nodes at each crossing point
3. Split original edges into segments at the crossing
4. Remove degenerate (zero-length) edges

This is critical for realistic mechanical simulation — welded crossings
transfer load between fibers.

In [ ]:
# Create a network with crossings
gen = RegularNetworkGenerator(side_length=10, num_points_per_side=2, tiling=3)
G_raw = gen.generate()

# Before welding
print(f"Before weld: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")

# Find intersections (without modifying graph)
intersections = find_intersections(G_raw)
print(f"Detected {len(intersections)} crossing points")

# Weld the graph
G_welded = weld_graph(G_raw)
print(f"After weld:  {G_welded.number_of_nodes()} nodes, {G_welded.number_of_edges()} edges")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, G, title in zip(axes, [G_raw, G_welded], ["Raw (no weld)", "Welded (junctions added)"]):
    pos = nx.get_node_attributes(G, 'pos')
    for u, v in G.edges():
        x = [pos[u][0], pos[v][0]]
        y = [pos[u][1], pos[v][1]]
        ax.plot(x, y, 'k-', linewidth=1.5)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.savefig('weld_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Visualization: Professional Fiber Network Rendering

The `plot_graph()` function follows the visualization principle from
nonwoven microscopy: **show fibers (edges), hide junctions (nodes)**.

This produces clean, publication-quality images that focus on
the fiber architecture rather than graph topology clutter.

In [ ]:
# Generate and weld a larger network
gen = RegularNetworkGenerator(side_length=10, num_points_per_side=3, tiling=4, perturbations=[(0.5, 0.5)])
G = gen.generate()
G_welded = weld_graph(G)

# Plot with default settings (edges only, no nodes)
fig, ax = plot_graph(G_welded, title="Fiber Network (Welded)", figsize=(10, 10))
plt.savefig('fiber_network_clean.png', dpi=200, bbox_inches='tight')
plt.show()

# Plot with nodes visible (for debugging)
fig, ax = plot_graph(G_welded, show_nodes=True, title="With Junction Nodes", figsize=(10, 10))
plt.savefig('fiber_network_with_nodes.png', dpi=200, bbox_inches='tight')
plt.show()

# Structure statistics
fig, axes = plot_structure_stats(G_welded)
plt.savefig('structure_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Inspecting the Data Structure

Understanding the graph representation is key to working with FiberNet v2.0.

Each graph is a `networkx.Graph` with:
- **Nodes**: have a `pos` attribute (x, y) or (x, y, z)
- **Edges**: represent fiber segments between junctions/endpoints
- **Graph attributes**: optional metadata like `canvas_size`, `generator`

In [ ]:
# Examine the graph structure
G = RegularNetworkGenerator(side_length=10, num_points_per_side=1, tiling=2).generate()
G_welded = weld_graph(G)

# Node positions
pos = nx.get_node_attributes(G_welded, 'pos')
print("Sample nodes (id -> position):")
for i, (node, p) in enumerate(list(pos.items())[:5]):
    print(f"  Node {node}: ({p[0]:.2f}, {p[1]:.2f})")

# Edge list
print(f"
Sample edges (first 5 of {G_welded.number_of_edges()}):")
for u, v in list(G_welded.edges())[:5]:
    print(f"  {u} -> {v}")

# Degree distribution
degrees = [d for n, d in G_welded.degree()]
print(f"
Degree statistics:")
print(f"  Mean degree: {np.mean(degrees):.2f}")
print(f"  Max degree:  {max(degrees)}")
print(f"  Min degree:  {min(degrees)}")

# Save and reload
save_graph_json(G_welded, 'my_network.json')
print("
Saved to my_network.json")

# Load back
net = load_graph_json('my_network.json')
print(f"Loaded: FiberNetwork with {len(net.fibers)} fibers")

## 8. Feature Extraction: 94-Dimensional Descriptor

The `GraphFeatureExtractor` computes a comprehensive 94-dimensional
feature vector from any fiber network graph:

| Group | Count | Description |
|-------|-------|-------------|
| **Structural** | 34 | Node/edge counts, degree stats, density, connectivity |
| **Pore** | 18 | Pore size distribution, porosity, specific surface |
| **Contact** | 42 | Weld/crosslink density, coordination, anisotropy |

This feature vector serves as input for ML models (surrogate modeling,
structure-property prediction, optimization).

In [ ]:
# Extract features from a welded network
gen = RegularNetworkGenerator(side_length=10, num_points_per_side=2, tiling=4, perturbations=[(0.5, 0.5)])
G = gen.generate()
G_welded = weld_graph(G)

extractor = GraphFeatureExtractor(canvas_size=256)
features = extractor.extract(G_welded)

print(f"Extracted {len(features)} features:
")

# Group by category
groups = {
    'Structural': ['n_node', 'n_edge', 'mean_degree', 'max_degree', 'density', 'n_components'],
    'Pore': ['porosity', 'mean_pore_area', 'max_pore_area', 'pore_count', 'specific_surface'],
    'Contact': ['n_weld', 'weld_density', 'mean_coordination', 'anisotropy'],
}

for group, keys in groups.items():
    print(f"
--- {group} Features ---")
    for k in keys:
        if k in features:
            print(f"  {k}: {features[k]:.4f}")

# Convenience function
features_quick = fn.extract_features(G_welded, canvas_size=256)
assert len(features_quick) == 94
print(f"
Quick extraction: {len(features_quick)} features ✓")

## 9. Parametric Sweep: Structure-Property Relationships

A common workflow is to sweep over perturbation parameters and
observe how structural features change. This creates datasets
for training ML surrogate models.

In [ ]:
# Sweep perturbation magnitude
dx_values = np.linspace(0, 2.0, 10)
results = []

for dx in dx_values:
    gen = RegularNetworkGenerator(
        side_length=10,
        num_points_per_side=2,
        tiling=4,
        perturbations=[(dx, dx)],
    )
    G = gen.generate()
    G_welded = weld_graph(G)
    
    features = fn.extract_features(G_welded, canvas_size=256)
    features['perturbation'] = dx
    results.append(features)

# Plot key metrics vs perturbation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

perturbations = [r['perturbation'] for r in results]
axes[0].plot(perturbations, [r['n_weld'] for r in results], 'bo-')
axes[0].set_xlabel('Perturbation (dx)')
axes[0].set_ylabel('Number of welds')
axes[0].set_title('Weld Count vs Disorder')

axes[1].plot(perturbations, [r['porosity'] for r in results], 'rs-')
axes[1].set_xlabel('Perturbation (dx)')
axes[1].set_ylabel('Porosity')
axes[1].set_title('Porosity vs Disorder')

axes[2].plot(perturbations, [r['mean_degree'] for r in results], 'g^-')
axes[2].set_xlabel('Perturbation (dx)')
axes[2].set_ylabel('Mean Degree')
axes[2].set_title('Connectivity vs Disorder')

plt.tight_layout()
plt.savefig('parametric_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Sweep complete: {len(results)} structures characterized")

## 10. ML Surrogate Model: Predicting Properties from Structure

With the 94-dimensional feature vectors, we can train ML models
to predict material properties without running expensive FEM simulations.

This example demonstrates a simple Random Forest regressor to predict
weld density from perturbation parameters.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# Generate training dataset (larger sweep)
n_samples = 50
X_data = []
y_data = []

for i in range(n_samples):
    dx = np.random.uniform(0, 2.5)
    dy = np.random.uniform(0, 2.5)
    n_pts = np.random.choice([1, 2, 3])
    
    gen = RegularNetworkGenerator(
        side_length=10,
        num_points_per_side=n_pts,
        tiling=3,
        perturbations=[(dx, dy)],
    )
    G = gen.generate()
    G_welded = weld_graph(G)
    features = fn.extract_features(G_welded, canvas_size=128)
    
    # Use subset of features as input
    feat_vec = [features[k] for k in sorted(features.keys()) if isinstance(features[k], (int, float))]
    X_data.append(feat_vec)
    y_data.append(features['n_weld'])

X = np.array(X_data)
y = np.array(y_data)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Random Forest Surrogate Model:")
print(f"  Training samples: {len(X_train)}")
print(f"  Test R²: {r2:.4f}")
print(f"  Test MAE: {mae:.2f} welds")

# Plot predictions
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.7, edgecolors='k')
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual weld count')
ax.set_ylabel('Predicted weld count')
ax.set_title(f'ML Surrogate: R²={r2:.3f}, MAE={mae:.2f}')
ax.legend()
plt.savefig('ml_surrogate.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. ZigZag Generator: Alternative Fiber Architecture

The `ZigZagGenerator` creates zigzag-pattern fiber networks
with configurable mirroring and tiling. This represents a different
class of manufactured nonwoven structures.

In [ ]:
# Generate ZigZag networks with different parameters
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

configs = [
    {"n_cols": 3, "n_rows": 3, "mirror_x": False, "mirror_y": False},
    {"n_cols": 4, "n_rows": 4, "mirror_x": True, "mirror_y": False},
    {"n_cols": 5, "n_rows": 5, "mirror_x": False, "mirror_y": True},
    {"n_cols": 3, "n_rows": 3, "mirror_x": True, "mirror_y": True},
    {"n_cols": 6, "n_rows": 4, "mirror_x": True, "mirror_y": True},
    {"n_cols": 4, "n_rows": 6, "mirror_x": False, "mirror_y": True},
]

for i, cfg in enumerate(configs):
    gen = ZigZagGenerator(**cfg)
    G = gen.generate()
    
    ax = axes[i // 3][i % 3]
    pos = nx.get_node_attributes(G, 'pos')
    for u, v in G.edges():
        x = [pos[u][0], pos[v][0]]
        y = [pos[u][1], pos[v][1]]
        ax.plot(x, y, 'k-', linewidth=1.2)
    
    label = f"cols={cfg['n_cols']}, rows={cfg['n_rows']}, mx={cfg['mirror_x']}, my={cfg['mirror_y']}"
    ax.set_title(f"{label}
({G.number_of_nodes()}n, {G.number_of_edges()}e)")
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.savefig('zigzag_variants.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Structure Comparison: Regular vs ZigZag

Compare the two generator types side-by-side with matching parameters.

In [ ]:
# Generate comparable networks
G_regular = RegularNetworkGenerator(side_length=10, num_points_per_side=2, tiling=4).generate()
G_zigzag = ZigZagGenerator(n_cols=5, n_rows=5, mirror_x=True, mirror_y=True).generate()

# Weld both
G_reg_w = weld_graph(G_regular)
G_zig_w = weld_graph(G_zigzag)

# Compare features
feat_reg = fn.extract_features(G_reg_w, canvas_size=256)
feat_zig = fn.extract_features(G_zig_w, canvas_size=256)

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, G, name, feat in zip(axes, [G_reg_w, G_zig_w], ["Regular", "ZigZag"], [feat_reg, feat_zig]):
    pos = nx.get_node_attributes(G, 'pos')
    for u, v in G.edges():
        x = [pos[u][0], pos[v][0]]
        y = [pos[u][1], pos[v][1]]
        ax.plot(x, y, 'k-', linewidth=1.0)
    ax.set_title(f"{name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges
"
                 f"Porosity={feat['porosity']:.3f}, Welds={feat['n_weld']}")
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.savefig('structure_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature heatmap comparison
key_features = ['n_node', 'n_edge', 'mean_degree', 'density', 'porosity', 'n_weld', 
                'weld_density', 'mean_coordination', 'anisotropy']
reg_vals = [feat_reg.get(k, 0) for k in key_features]
zig_vals = [feat_zig.get(k, 0) for k in key_features]

# Normalize for comparison
max_vals = [max(r, z, 1e-10) for r, z in zip(reg_vals, zig_vals)]
reg_norm = [r / m for r, m in zip(reg_vals, max_vals)]
zig_norm = [z / m for z, m in zip(zig_vals, max_vals)]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(key_features))
ax.bar(x - 0.2, reg_norm, 0.4, label='Regular', color='steelblue')
ax.bar(x + 0.2, zig_norm, 0.4, label='ZigZag', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(key_features, rotation=45, ha='right')
ax.set_ylabel('Normalized Value')
ax.set_title('Feature Comparison: Regular vs ZigZag')
ax.legend()
plt.tight_layout()
plt.savefig('feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Saving and Loading Graph Data

FiberNet v2.0 uses a JSON graph format compatible with D3.js and other tools.

**JSON Schema:**


NetworkX graphs can also be converted bidirectionally with `FiberNetwork` objects.

In [ ]:
# Generate, weld, and save
gen = RegularNetworkGenerator(side_length=10, num_points_per_side=2, tiling=3)
G = gen.generate()
G_welded = weld_graph(G)

# Save as JSON
save_graph_json(G_welded, 'welded_network.json')
print("Saved welded network to JSON")

# Load back as FiberNetwork
net = load_graph_json('welded_network.json')
print(f"Loaded: {len(net.fibers)} fibers, material={net.material}")

# Convert to/from NetworkX
G_nx = fn.to_networkx(net)
print(f"NetworkX graph: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")

net_back = fn.from_networkx(G_nx)
print(f"Roundtrip: {len(net_back.fibers)} fibers")

# Feature extraction works on both types
feat_from_graph = fn.extract_features(G_welded, canvas_size=256)
feat_from_net = fn.extract_features(net, canvas_size=256)
print(f"Features from graph: {len(feat_from_graph)} dims")
print(f"Features from FiberNetwork: {len(feat_from_net)} dims")

## 14. Complete Research Pipeline

Here is the full workflow from structure generation to property prediction,
demonstrating how all components work together.

In [ ]:
import time

print("=" * 60)
print("FiberNet v2.0 Complete Research Pipeline")
print("=" * 60)

# Step 1: Generate diverse structures
print("
[1/5] Generating diverse fiber networks...")
structures = {}
t0 = time.time()

for dx in [0.0, 0.5, 1.0, 1.5, 2.0]:
    for n_pts in [1, 2, 3]:
        name = f"regular_dx{dx}_npts{n_pts}"
        gen = RegularNetworkGenerator(
            side_length=10, num_points_per_side=n_pts,
            tiling=4, perturbations=[(dx, dx)]
        )
        G = gen.generate()
        G_welded = weld_graph(G)
        structures[name] = G_welded

# Add zigzag structures
for cols in [3, 4, 5]:
    name = f"zigzag_{cols}x{cols}"
    G = ZigZagGenerator(n_cols=cols, n_rows=cols, mirror_x=True, mirror_y=True).generate()
    structures[name] = weld_graph(G)

print(f"  Generated {len(structures)} structures in {time.time()-t0:.1f}s")

# Step 2: Extract features
print("
[2/5] Extracting 94-dim features...")
t0 = time.time()
all_features = {}
for name, G in structures.items():
    all_features[name] = fn.extract_features(G, canvas_size=256)
print(f"  Extracted features in {time.time()-t0:.1f}s")

# Step 3: Build feature matrix
print("
[3/5] Building feature matrix...")
sorted_keys = sorted(all_features[list(all_features.keys())[0]].keys())
X = np.array([[all_features[name].get(k, 0) for k in sorted_keys] 
               for name in all_features])
names = list(all_features.keys())
print(f"  Feature matrix: {X.shape}")

# Step 4: Dimensionality reduction (PCA)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['steelblue' if 'regular' in n else 'coral' for n in names]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, s=100, edgecolors='k', alpha=0.7)
for i, name in enumerate(names):
    ax.annotate(name.split('_')[-1], (X_pca[i, 0], X_pca[i, 1]), fontsize=8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA of Fiber Network Feature Space')
plt.savefig('pca_feature_space.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 5: Summary
print("
[4/5] Structure summary:")
print(f"  {'Name':<30} {'Nodes':>6} {'Edges':>6} {'Welds':>6} {'Porosity':>9}")
print(f"  {'-'*57}")
for name in list(structures.keys())[:5]:
    G = structures[name]
    f = all_features[name]
    print(f"  {name:<30} {G.number_of_nodes():>6} {G.number_of_edges():>6} {f['n_weld']:>6} {f['porosity']:>9.4f}")
print(f"  ... and {len(structures)-5} more structures")

print(f"
[5/5] Pipeline complete!")
print(f"  Total structures: {len(structures)}")
print(f"  Feature dimensions: {X.shape[1]}")
print(f"  Ready for ML/RL downstream tasks")

## 15. API Reference

### Graph Operations (`fibernet.graph`)

| Function | Description |
|----------|-------------|
| `weld_graph(G)` | Detect edge crossings, insert junction nodes, split edges |
| `find_intersections(G)` | Return list of crossing points without modifying graph |
| `merge_coincident_nodes(G, tol)` | Merge nodes closer than tolerance |
| `save_graph_json(G, path)` | Save graph to JSON file |
| `load_graph_json(path)` | Load JSON as FiberNetwork |
| `to_networkx(net)` | Convert FiberNetwork → nx.Graph |
| `from_networkx(G)` | Convert nx.Graph → FiberNetwork |

### Generators (`fibernet.gen`)

| Class | Key Parameters |
|-------|---------------|
| `RegularNetworkGenerator` | `side_length`, `num_points_per_side`, `perturbations`, `tiling` |
| `ZigZagGenerator` | `n_cols`, `n_rows`, `mirror_x`, `mirror_y` |

### Feature Extraction (`fibernet.analysis`)

| Class/Function | Description |
|----------------|-------------|
| `GraphFeatureExtractor(canvas_size)` | 94-dim feature vector from graph |
| `fn.extract_features(G, canvas_size)` | Convenience function (accepts nx.Graph or FiberNetwork) |

### Visualization (`fibernet.viz`)

| Function | Description |
|----------|-------------|
| `plot_graph(G, show_nodes=False)` | Clean fiber-edge plot (publication-ready) |
| `plot_graph_comparison(graphs, labels)` | Side-by-side comparison |
| `plot_structure_stats(G)` | Degree distribution, edge length histogram, connectivity |

### Top-Level Shortcuts (`fibernet`)

All graph API functions are available at the top level:
```python
import fibernet as fn
fn.RegularNetworkGenerator(...)
fn.ZigZagGenerator(...)
fn.weld_graph(...)
fn.extract_features(...)
fn.plot_graph(...)
fn.save_graph_json(...)
fn.load_graph_json(...)
```

## Next Steps

- **02_mechanical_simulation.ipynb** — Run FEM on welded networks
- **03_machine_learning.ipynb** — Train property prediction models
- **04_reinforcement_validation.ipynb** — RL-based structure optimization

For questions and contributions, visit [github.com/GellmanSparrowS/fibernet](https://github.com/GellmanSparrowS/fibernet)